In [ ]:
!pip install -U diffusers transformers accelerate torch torchvision -q
!pip uninstall -y torchaudio -q

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("تم تسجيل الدخول ✅")

تم تسجيل الدخول ✅


In [3]:
import requests
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()
print(f"عدد البريفات: {len(briefs)}")

عدد البريفات: 30


In [ ]:
!pip install -U bitsandbytes -q

In [ ]:
import torch, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffusersBnBConfig
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBnBConfig

model_id = "black-forest-labs/FLUX.1-schnell"

# تكميم الـ transformer (أكبر جزء بالموديل)
transformer_quant_config = DiffusersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
transformer = FluxTransformer2DModel.from_pretrained(
    model_id, subfolder="transformer",
    quantization_config=transformer_quant_config,
    dtype=torch.bfloat16,
)

# تكميم الـ T5 text encoder (تاني أكبر جزء)
text_encoder_quant_config = TransformersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
text_encoder_2 = T5EncoderModel.from_pretrained(
    model_id, subfolder="text_encoder_2",
    quantization_config=text_encoder_quant_config,
    dtype=torch.bfloat16,
)

# تجميع البايبلاين بالأجزاء المكمّمة
pipe = FluxPipeline.from_pretrained(
    model_id,
    transformer=transformer,
    text_encoder_2=text_encoder_2,
    dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()

print("✅ الموديل جاهز بنسخة مكمّمة")